In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")

In [2]:
import os
import pickle
import glob
from collections import defaultdict

from tvm import auto_scheduler

# 모듈 경로
import sys
sys.path.insert(0, "/root/work/tvm-ansor/gallery/constrained_gen")

from modules.record_loader import (
    load_and_register_network,
    group_by_sketches_from_json,
    state_sketch_fingerprint,
    load_records_from_dir,
    group_records_by_wkey_and_sketch,
)
from modules.param_manager import build_symbolic_state, verify_symbolic_state

# ─── 경로 ───
NETWORK_INFO_DIR = "/root/work/tvm-ansor/gallery/dataset/network_info"
NETWORKS_DIR = "/root/work/tvm-ansor/gallery/dataset/to_measure_networks"

# ─── 1) all_tasks.pkl 로드 ───
all_tasks = pickle.load(open(os.path.join(NETWORK_INFO_DIR, "all_tasks.pkl"), "rb"))
for task in all_tasks:
    auto_scheduler.workload_registry.register_workload_tensors(
        task.workload_key, task.compute_dag.tensors)
print(f"Loaded {len(all_tasks)} tasks from all_tasks.pkl")

# ─── 2) to_measure_networks 전체 JSON 로드 ───
network_dirs = sorted([
    d for d in os.listdir(NETWORKS_DIR)
    if os.path.isdir(os.path.join(NETWORKS_DIR, d))
])
print(f"Found {len(network_dirs)} network directories")

# 모든 네트워크의 records를 하나로 합침
all_records = {}  # {wkey: [(inp, res), ...]}
for net_dir in network_dirs:
    records_path = os.path.join(NETWORKS_DIR, net_dir)
    recs = load_records_from_dir(all_tasks, records_path)
    for wkey, rec_list in recs.items():
        all_records.setdefault(wkey, []).extend(rec_list)

print(f"Loaded records for {len(all_records)} unique workload keys, "
      f"total {sum(len(v) for v in all_records.values())} records")

# ─── 3) sketch 그룹핑 ───
grouped = group_records_by_wkey_and_sketch(all_records)

total_sketches = sum(len(sd) for sd in grouped.values())
print(f"Grouped into {total_sketches} unique sketches across {len(grouped)} workload keys")

Loaded 849 tasks from all_tasks.pkl
Found 56 network directories
Loaded records for 849 unique workload keys, total 2023755 records
Loaded records for 849 unique workload keys, total 2023755 records
Grouped into 912 unique sketches across 849 workload keys
Grouped into 912 unique sketches across 849 workload keys


In [9]:
# ─── 모듈 리로드 (param_manager 수정 반영) ───
import importlib
import modules.sym_types
import modules.symbolic_state
import modules.transform_applier
import modules.param_manager
importlib.reload(modules.sym_types)
importlib.reload(modules.symbolic_state)
importlib.reload(modules.transform_applier)
importlib.reload(modules.param_manager)
from modules.param_manager import build_symbolic_state, verify_symbolic_state
print("Modules reloaded")

Modules reloaded


In [4]:
# 지금까지 실패 현황 확인
print(f"sketch_idx={sketch_idx}, pass={pass_count}, fail={fail_count}, build_err={build_error_count}")
print(f"\n=== fail_details ({len(fail_details)} items) ===")
for i, (wkey, fp, summary) in enumerate(fail_details):
    task = wkey_to_task.get(wkey)
    desc = task.desc if task else wkey[:40]
    print(f"\n[{i+1}] {desc}")
    print(f"    {summary}")

sketch_idx=128, pass=107, fail=20, build_err=0

=== fail_details (20 items) ===

[1] vm_mod_fused_mean
    10/10 failed: FAIL(ext_mm=1/7)

[2] vm_mod_fused_variance
    10/10 failed: FAIL(ext_mm=1/13)

[3] vm_mod_fused_mean
    10/10 failed: FAIL(ext_mm=1/7)

[4] vm_mod_fused_variance
    10/10 failed: FAIL(ext_mm=1/13)

[5] vm_mod_fused_mean
    10/10 failed: FAIL(ext_mm=1/7)

[6] vm_mod_fused_variance
    10/10 failed: FAIL(ext_mm=1/13)

[7] vm_mod_fused_mean
    15/15 failed: FAIL(ext_mm=1/7)

[8] vm_mod_fused_variance
    15/15 failed: FAIL(ext_mm=1/13)

[9] vm_mod_fused_mean
    15/15 failed: FAIL(ext_mm=1/7)

[10] vm_mod_fused_variance
    15/15 failed: FAIL(ext_mm=1/13)

[11] vm_mod_fused_mean
    15/15 failed: FAIL(ext_mm=1/7)

[12] vm_mod_fused_variance
    15/15 failed: FAIL(ext_mm=1/13)

[13] vm_mod_fused_mean
    10/10 failed: FAIL(ext_mm=1/7)

[14] vm_mod_fused_variance
    10/10 failed: FAIL(ext_mm=1/13)

[15] vm_mod_fused_mean
    10/10 failed: FAIL(ext_mm=1/7)

[16] vm_

In [5]:
# 실패 케이스 상세 분석: fused_mean (첫 번째)
wkey1, fp1, _ = fail_details[0]
task1 = wkey_to_task[wkey1]
recs1 = grouped[wkey1][fp1]
base_inp1, _ = recs1[0]
sym1 = build_symbolic_state(task1.compute_dag, base_inp1.state)

# verbose로 첫 번째 state 검증
fail_inp, _ = recs1[0]
ok, summary = verify_symbolic_state(task1, fail_inp.state, sym1, verbose=True)
print(f"\n{task1.desc}: {summary}")


vm_mod_fused_mean: FAIL(ext_mm=1/7)
  EXT  s2.i1('ax2.1'): real=32 sym=min(sp_0_0,1)→eval=1


In [6]:
# 실패 케이스 심층 분석: fused_mean의 state 구조 확인
wkey1, fp1, _ = fail_details[0]
task1 = wkey_to_task[wkey1]
recs1 = grouped[wkey1][fp1]

print(f"Task: {task1.desc}")
print(f"Records in this sketch: {len(recs1)}")

# 원본 compute_dag의 axis 확인
print("\n=== compute_dag ops ===")
for sid, op in enumerate(task1.compute_dag.ops):
    if hasattr(op, 'axis'):
        axes = [(str(a.var.name), int(a.dom.extent) if a.dom else None) for a in op.axis]
        raxes = [(str(a.var.name), int(a.dom.extent) if a.dom else None) for a in op.reduce_axis]
        print(f"  stage {sid}: {op.name}")
        print(f"    axis: {axes}")
        if raxes:
            print(f"    reduce_axis: {raxes}")
    else:
        print(f"  stage {sid}: {op.name} (placeholder)")

# transform steps 확인
base_state = recs1[0][0].state
print(f"\n=== Transform steps ({len(base_state.transform_steps)}) ===")
for i, step in enumerate(base_state.transform_steps):
    tk = step.type_key.split(".")[-1]
    if tk == "SplitStep":
        lengths = [int(l) if l is not None else None for l in step.lengths]
        ext = int(step.extent) if step.extent is not None else None
        print(f"  [{i}] {tk} s{step.stage_id}.i{step.iter_id} lengths={lengths} extent={ext} inner_to_outer={step.inner_to_outer}")
    elif tk == "FuseStep":
        fused = [int(x) for x in step.fused_ids]
        print(f"  [{i}] {tk} s{step.stage_id} fused_ids={fused}")
    elif tk == "AnnotationStep":
        print(f"  [{i}] {tk} s{step.stage_id}.i{step.iter_id} ann={int(step.annotation)}")
    elif tk == "ReorderStep":
        after = [int(x) for x in step.after_ids]
        print(f"  [{i}] {tk} s{step.stage_id} after_ids={after}")
    elif tk == "PragmaStep":
        print(f"  [{i}] {tk} s{step.stage_id}.i{step.iter_id} pragma={step.pragma_type}")
    elif tk == "ComputeAtStep":
        print(f"  [{i}] {tk} s{step.stage_id} -> s{step.target_stage_id}.i{step.target_iter_id}")
    elif tk == "ComputeInlineStep":
        print(f"  [{i}] {tk} s{step.stage_id}")
    elif tk == "ComputeRootStep":
        print(f"  [{i}] {tk} s{step.stage_id}")
    else:
        print(f"  [{i}] {tk} s{step.stage_id}")

# SymbolicState 구조 확인
sym1 = build_symbolic_state(task1.compute_dag, base_state)
print(f"\n=== SymbolicState (sym_map) ===")
for k, v in sym1.sym_map.items():
    print(f"  {k} = {v}")
print(f"\n=== SymbolicState stages ===")
for sid, stage in enumerate(sym1.stages):
    if stage.op_type == 'placeholder':
        continue
    print(f"\n  stage {sid}: {stage.op_name} (compute_at={stage.compute_at})")
    for iid, it in enumerate(stage.iters):
        print(f"    i{iid}: {it}")

Task: vm_mod_fused_mean
Records in this sketch: 10

=== compute_dag ops ===
  stage 0: p0 (placeholder)
  stage 1: p0_red
    axis: [('ax0', 1), ('ax1', 128), ('ax2', 1)]
    reduce_axis: [('k2', 768)]
  stage 2: T_divide
    axis: [('ax0', 1), ('ax1', 128), ('ax2', 1)]

=== Transform steps (8) ===
  [0] SplitStep s2.i2 lengths=[32] extent=1 inner_to_outer=1
  [1] AnnotationStep s2.i3 ann=6
  [2] FollowSplitStep s1
  [3] AnnotationStep s1.i4 ann=6
  [4] ComputeAtStep s1 -> s2.i2
  [5] FuseStep s2 fused_ids=[0, 1, 2]
  [6] AnnotationStep s2.i0 ann=5
  [7] PragmaStep s1.i0 pragma=auto_unroll_max_step$1024

=== SymbolicState (sym_map) ===
  sp_0_0 = 32
  ur_7 = 1024

=== SymbolicState stages ===

  stage 1: p0_red (compute_at=2)
    i0: for ax0 (0,min(sp_0_0,1))
    i1: for ax1 (0,min(sp_0_0,1))
    i2: for ax2 (0,min(sp_0_0,1))
    i3: for k2.0 (0,ceil(768/(min(sp_0_0,768))))
    i4: threadIdx.x k2.1 (0,min(sp_0_0,768))

  stage 2: T_divide (compute_at=0)
    i0: blockIdx.x ax0@ax1@ax2.0

In [7]:
# TVM InferBound vs Symbolic: 어떤 쪽이 올바른지 확인
# InferBound 결과를 직접 보자
bounded = task1.compute_dag.infer_bound_from_state(recs1[0][0].state)
print("=== InferBound stage 2 (T_divide) ===")
for iid, it in enumerate(bounded.stages[2].iters):
    ext = int(it.range.extent) if it.range else None
    print(f"  i{iid} '{it.name}': extent={ext} ann={int(it.annotation)}")

print("\n=== InferBound stage 1 (p0_red) ===")
for iid, it in enumerate(bounded.stages[1].iters):
    ext = int(it.range.extent) if it.range else None
    print(f"  i{iid} '{it.name}': extent={ext} ann={int(it.annotation)}")

# 다른 state도 비교 (sp_0_0 값이 다른 경우)
print("\n=== 다른 state들의 sp_0_0 값과 InferBound ax2.1 extent ===")
for ri, (inp, res) in enumerate(recs1[:5]):
    sp_val = int(inp.state.transform_steps[0].lengths[0])
    bounded2 = task1.compute_dag.infer_bound_from_state(inp.state)
    ext_ax2_1 = int(bounded2.stages[2].iters[1].range.extent)
    print(f"  rec[{ri}] sp_0_0={sp_val}, InferBound ax2.1={ext_ax2_1}, actual_min={min(sp_val, 1)}")

=== InferBound stage 2 (T_divide) ===
  i0 'ax0@ax1@ax2.0@': extent=128 ann=5
  i1 'ax2.1': extent=32 ann=6

=== InferBound stage 1 (p0_red) ===
  i0 'ax0': extent=1 ann=0
  i1 'ax1': extent=1 ann=0
  i2 'ax2': extent=1 ann=0
  i3 'k2.0': extent=24 ann=0
  i4 'k2.1': extent=32 ann=6

=== 다른 state들의 sp_0_0 값과 InferBound ax2.1 extent ===
  rec[0] sp_0_0=32, InferBound ax2.1=32, actual_min=1
  rec[1] sp_0_0=32, InferBound ax2.1=32, actual_min=1
  rec[2] sp_0_0=32, InferBound ax2.1=32, actual_min=1
  rec[3] sp_0_0=32, InferBound ax2.1=32, actual_min=1
  rec[4] sp_0_0=32, InferBound ax2.1=32, actual_min=1


In [8]:
# conv2d_transpose 실패 케이스도 분석
wkey_ct, fp_ct, summary_ct = fail_details[18]  # conv2d_transpose_3
task_ct = wkey_to_task[wkey_ct]
recs_ct = grouped[wkey_ct][fp_ct]

print(f"Task: {task_ct.desc}")
print(f"Records in this sketch: {len(recs_ct)}")
print(f"Summary: {summary_ct}")

# verbose 검증
base_ct = recs_ct[0][0]
sym_ct = build_symbolic_state(task_ct.compute_dag, base_ct.state)
# 실패하는 첫 번째 state 찾기
for ri, (inp_ct, res_ct) in enumerate(recs_ct):
    ok_ct, sum_ct = verify_symbolic_state(task_ct, inp_ct.state, sym_ct, verbose=True)
    if not ok_ct:
        print(f"\nFailed rec[{ri}]:")
        print(sum_ct)
        break

Task: vm_mod_fused_nn_conv2d_transpose_3
Records in this sketch: 2000
Summary: 748/2000 failed: FAIL(ext_mm=1/46)

Failed rec[1]:
FAIL(ext_mm=1/46)
  EXT  s4.i0('ax0@ax1@ax2@ax3@.0.0'): real=4 sym=ceil(ceil(sp_1_0*sp_1_1*sp_1_2*sp_1_3*sp_2_0*sp_2_1*sp_2_2*sp_2_3*sp_6_0*sp_6_1*sp_5_0*sp_5_1/(min(sp_27_0,sp_1_0*sp_1_1*sp_1_2*sp_1_3*sp_2_0*sp_2_1*sp_2_2*sp_2_3*sp_6_0*sp_6_1*sp_5_0*sp_5_1)))/(sp_4_1*sp_3_1*sp_2_1*sp_1_1))→eval=1


In [ ]:
# ─── 전체 912 스케치 배치 검증 (수정된 verify_symbolic_state) ───
wkey_to_task = {t.workload_key: t for t in all_tasks}

# 모든 (wkey, sketch_fp, recs) 항목을 flat list로 구성
all_sketch_items = []
for wkey, sketch_dict in grouped.items():
    task = wkey_to_task.get(wkey)
    if task is None:
        continue
    for sketch_fp, recs in sketch_dict.items():
        all_sketch_items.append((wkey, sketch_fp, recs))

total_sketches = len(all_sketch_items)
print(f"Total sketches to verify: {total_sketches}")

BATCH_SIZE = 100
pass_count = 0
fail_count = 0
build_error_count = 0
tight_count = 0
fail_details = []

for batch_start in range(0, total_sketches, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, total_sketches)
    batch_pass = 0
    batch_fail = 0
    batch_build_err = 0
    batch_tight = 0

    for idx in range(batch_start, batch_end):
        wkey, sketch_fp, recs = all_sketch_items[idx]
        task = wkey_to_task[wkey]

        # symbolic state 생성
        base_inp, base_res = recs[0]
        try:
            sym_state = build_symbolic_state(task.compute_dag, base_inp.state)
        except Exception as e:
            build_error_count += 1
            batch_build_err += 1
            fail_details.append((wkey, sketch_fp, f"BUILD_ERROR: {e}"))
            continue

        # 검증
        sketch_pass = 0
        sketch_fail = 0
        sketch_tight = 0
        first_fail_summary = None

        for inp, res in recs:
            ok, summary = verify_symbolic_state(task, inp.state, sym_state, verbose=False)
            if ok:
                sketch_pass += 1
                if "ext_tight" in summary:
                    sketch_tight += 1
            else:
                sketch_fail += 1
                if first_fail_summary is None:
                    first_fail_summary = summary

        if sketch_fail == 0:
            batch_pass += 1
            if sketch_tight > 0:
                batch_tight += 1
        else:
            batch_fail += 1
            fail_details.append((wkey, sketch_fp,
                f"{sketch_fail}/{len(recs)} failed: {first_fail_summary}"))

    pass_count += batch_pass
    fail_count += batch_fail
    tight_count += batch_tight

    print(f"  batch [{batch_start+1}-{batch_end}/{total_sketches}] "
          f"pass={batch_pass}(tight={batch_tight}) fail={batch_fail} build_err={batch_build_err}")

    # 이 배치에서 새로 발견된 실패 상세 출력
    new_fails = fail_details[-(batch_fail + batch_build_err):]
    for wkey, fp, summary in new_fails:
        task = wkey_to_task.get(wkey)
        desc = task.desc if task else wkey[:40]
        print(f"    ✗ {desc}: {summary}")

print(f"\n{'='*60}")
print(f"Total sketches: {total_sketches}")
print(f"  PASS:        {pass_count} (of which {tight_count} have tighter sym bounds)")
print(f"  FAIL:        {fail_count}")
print(f"  BUILD_ERROR: {build_error_count}")
print(f"{'='*60}")

Total sketches to verify: 912
  batch [1-100/912] pass=60(tight=16) fail=40 build_err=0
    ✗ vm_mod_fused_nn_dense_add_fast_tanh: 5976/6000 failed: FAIL(ext_mm=1/30)
    ✗ vm_mod_fused_nn_batch_matmul_2: 6000/6000 failed: FAIL(ext_mm=6/30)
    ✗ vm_mod_fused_nn_batch_matmul_1: 6000/6000 failed: FAIL(ext_mm=5/30)
    ✗ vm_mod_fused_nn_batch_matmul: 4000/4000 failed: FAIL(ext_mm=5/30)
    ✗ vm_mod_fused_nn_batch_matmul_4: 4000/4000 failed: FAIL(ext_mm=5/30)
    ✗ vm_mod_fused_nn_batch_matmul_3: 4000/4000 failed: FAIL(ext_mm=5/30)
    ✗ vm_mod_fused_nn_batch_matmul_2: 6000/6000 failed: FAIL(ext_mm=7/30)
    ✗ vm_mod_fused_nn_batch_matmul_1: 6000/6000 failed: FAIL(ext_mm=3/30)
    ✗ vm_mod_fused_nn_batch_matmul: 4000/4000 failed: FAIL(ext_mm=4/30)
    ✗ vm_mod_fused_nn_batch_matmul_4: 4000/4000 failed: FAIL(ext_mm=3/30)
    ✗ vm_mod_fused_nn_batch_matmul_3: 4000/4000 failed: FAIL(ext_mm=4/30)
    ✗ vm_mod_fused_nn_fast_softmax_broadcast_to: 1161/1485 failed: FAIL(ext_mm=3/23)
    ✗ vm_mod

KeyboardInterrupt: 

: 